In [ ]:
!pip install gradio langdetect groq -q

In [ ]:
import pickle
import numpy as np
import langdetect
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
from groq import Groq
import warnings
warnings.filterwarnings('ignore')

In [ ]:
save_dir = "/content/drive/MyDrive/ExAI/models"

with open(f"{save_dir}/base_models.pkl", "rb") as f:
    base_models = pickle.load(f)

with open(f"{save_dir}/meta_model.pkl", "rb") as f:
    meta_model = pickle.load(f)

with open(f"{save_dir}/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

feature_names = vectorizer.get_feature_names_out()
coefficients = base_models["LogReg"].coef_[0]
print("Models loaded successfully.")

Models loaded successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
GROQ_API_KEY = {"your-api-key"}
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq ready.")

Groq ready.


In [ ]:
def stack_predict_proba(texts):
    vecs = vectorizer.transform(texts)
    base_preds = np.column_stack([
        model.predict_proba(vecs)[:, 1]
        for model in base_models.values()
    ])
    return meta_model.predict_proba(base_preds)


def detect_and_translate(text):
    try:
        lang = langdetect.detect(text)
    except:
        lang = "en"

    if lang == "en":
        return text, False

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"Translate the following text to English. Reply with only the translated text, nothing else.\n\nText:\n{text}"}]
    )
    return response.choices[0].message.content.strip(), True


def get_fast_explanation(text, n=10):
    vec = vectorizer.transform([text])
    tfidf_scores = vec.toarray()[0]
    contributions = tfidf_scores * coefficients
    present_idx = np.where(tfidf_scores > 0)[0]

    if len(present_idx) == 0:
        return [], [], None

    sorted_idx = present_idx[np.argsort(np.abs(contributions[present_idx]))[-n:][::-1]]
    top_words = [feature_names[i] for i in sorted_idx]
    top_values = [contributions[i] for i in sorted_idx]

    colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in top_values]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(top_words[::-1], top_values[::-1], color=colors[::-1])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Contribution Score")
    ax.set_title("Top Words Influencing This Prediction\n(Red = toward Fake, Green = toward Real)")
    plt.tight_layout()

    return top_words, top_values, fig


def generate_llm_explanation(text, prediction, confidence, top_words, top_values, was_translated):
    word_contributions = ", ".join([
        f"'{w}' ({'fake' if v > 0 else 'real'})"
        for w, v in zip(top_words[:3], top_values[:3])
    ])
    lang_note = "The article was translated from another language. " if was_translated else ""
    prompt = f"""{lang_note}A news article was classified as {prediction} with {confidence:.1%} confidence.

Article (first 300 chars): {text[:300]}

Key words that influenced the decision: {word_contributions}

Write a 3-4 sentence explanation for a non-technical user that:
1. States the verdict and confidence clearly
2. Explains specifically which words were suspicious or trustworthy and what they suggest about the article
3. Gives a brief caution to verify with trusted sources

Be specific about the words — don't just say 'certain words were flagged', name them and explain why they're suspicious or trustworthy."""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

In [ ]:
def analyze_article(article_text):
    if not article_text or len(article_text.strip()) < 20:
        return "Please enter a longer article.", None, ""

    translated_text, was_translated = detect_and_translate(article_text)

    probs = stack_predict_proba([translated_text])[0]
    pred_label = "FAKE" if probs[1] > 0.5 else "REAL"
    confidence = max(probs)

    top_words, top_values, fig = get_fast_explanation(translated_text)

    llm_explanation = generate_llm_explanation(
        translated_text, pred_label, confidence,
        top_words, top_values, was_translated
    )

    translation_note = "\n_(Article was translated from another language before analysis.)_" if was_translated else ""
    result_text = f"## {pred_label}\n**Confidence:** {confidence:.1%}{translation_note}"

    return result_text, fig, llm_explanation

In [ ]:
with gr.Blocks(title="FakeSight", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🔍 FakeSight
    ### Explainable Fake News Detection
    Paste any news article below — in any language — and FakeSight will tell you
    whether it's likely **real or fake**, and explain *why*.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            article_input = gr.Textbox(
                label="Paste News Article Here",
                placeholder="Enter the news article text...",
                lines=12
            )
            analyze_btn = gr.Button("Analyze", variant="primary")

        with gr.Column(scale=1):
            result_output = gr.Markdown(label="Prediction")
            explanation_output = gr.Textbox(
                label="Plain Language Explanation",
                lines=6,
                interactive=False
            )

    shap_plot = gr.Plot(label="Word-Level Explanation")

    analyze_btn.click(
        fn=analyze_article,
        inputs=[article_input],
        outputs=[result_output, shap_plot, explanation_output]
    )

    gr.Markdown("""
    ---
    *FakeSight uses a stacking ensemble (XGBoost + LinearSVC + Logistic Regression)
    with word-level explainability and Gemini for multilingual support.*
    """)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afe225ceab1820218a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
